# 02: Valuing every touch

VAEP in one sentence: an action is worth how much it changes your team's chance of scoring, minus how much it changes the opponent's. A through ball that creates a 30% chance is worth about +0.25. A sideways pass at halfway is worth about nothing. A goal is worth 1 minus the chance it came from.

In [1]:
from effc.vaep.features import featurize_match, K
from effc.data.statsbomb import load_events, list_matches
from effc.data.download import TARGET

mid = int(list_matches(TARGET)['match_id'].iloc[0])
df = featurize_match(load_events(mid), match_id=mid)
print(f"{len(df)} actions, K={K}")
print(df[['player','action','start_x','dist_start','scores_incl','scores_excl']].head(5).to_string(index=False))

1907 actions, K=10
          player action  start_x  dist_start  scores_incl  scores_excl
   Kevin Kuranyi   Pass     61.0   44.420828          0.0          0.0
   Kevin Volland  Carry     60.0   45.195575          0.0          0.0
   Kevin Volland   Pass     59.1   46.171528          0.0          0.0
Pirmin Schwegler  Carry     45.1   60.500083          0.0          0.0
Pirmin Schwegler   Pass     43.3   63.054659          0.0          0.0


Two label windows, and the asymmetry is the whole trick. The *including* window counts the action's own goal, so a shot's danger is visible before it. The *excluding* window does not, so the state after a goal is not poisoned by the goal that just happened. Getting this wrong gives goals zero value or keepers infinite value. I got it wrong twice before landing here.

In [2]:
import pickle, os
from effc.data.download import DATA_DIR
from effc.vaep.models import train_goal_models, brier_score, predict_proba

frames = pickle.load(open(os.path.join(DATA_DIR, 'cache', 'pool_frames.pkl'), 'rb'))
models = train_goal_models(frames, seed=0)
import pandas as pd
full = pd.concat(frames, ignore_index=True)
for lbl in ['scores_incl', 'concedes_incl']:
    p = predict_proba(models, full)[lbl]
    print(lbl, 'Brier:', round(brier_score(full[lbl].to_numpy(), p), 4))

scores_incl Brier: 0.0085


concedes_incl Brier: 0.0021


Brier scores around 0.008. For rare events that is well calibrated. Now value the match: watch what a goal, an assist, and a save are worth.

In [3]:
from effc.vaep.values import add_vaep

df = add_vaep(df, models)
goals = df[df['is_goal']]
print('GOALS:')
print(goals[['player','action','vaep','vaep_off']].to_string(index=False))
print()
saves = df[(df['action']=='Goal Keeper')].nlargest(3, 'vaep_def')
print('TOP SAVES (defensive value):')
print(saves[['player','vaep','vaep_def']].to_string(index=False))
print()
print('mean VAEP by action:')
print(df.groupby('action')['vaep'].mean().sort_values(ascending=False).round(4).head(8).to_string())

GOALS:
         player action     vaep  vaep_off
   Steven Zuber   Shot 0.786953  0.787929
Stefan Kießling   Shot 0.712747  0.713153
  Julian Brandt   Shot 0.914048  0.916573

TOP SAVES (defensive value):
        player     vaep  vaep_def
Oliver Baumann 0.651175  0.648164
Oliver Baumann 0.533742  0.533320
Oliver Baumann 0.492214  0.491331

mean VAEP by action:
action
Goal Keeper      0.0787
Shot             0.0312
Ball Recovery    0.0120
Clearance        0.0085
Interception     0.0072
Pass             0.0053
Carry            0.0046
Block            0.0036


Goals land around +0.7 (one goal minus the chance it came from). Saves land around +0.25. The keeper who concedes gets -0.7ish via the conceded-goal debit, which is what keeps keeper ratings honest. This all matches the theory, which is reassuring because the theory is the only thing I had to go on.